In [0]:
# ── CONFIG ───────────────────────────────────────────────────────────────────
 
CATALOG        = "clutchlytics"
BRONZE_TABLE   = f"{CATALOG}.bronze.raw_nhl_goalie_logs"
DIM_ATHLETES   = f"{CATALOG}.silver.dimAthletes"
DIM_GAMES      = f"{CATALOG}.silver.dimGames"
SILVER_TABLE   = f"{CATALOG}.silver.nhl_goalie_game_logs"
 
LEAGUE         = "nhl"
SEASON         = 2026
 
# True  → reprocess all goalies (use for first run or schema changes)
# False → incremental, only goalies where Bronze is newer than Silver
FULL_REFRESH   = True
 
print(f"Source       : {BRONZE_TABLE}")
print(f"dimAthletes  : {DIM_ATHLETES}")
print(f"dimGames     : {DIM_GAMES}")
print(f"Target       : {SILVER_TABLE}")
print(f"Season       : {SEASON} (hardcoded — meta.season bug in Bronze)")
print(f"Full refresh : {FULL_REFRESH}")

In [0]:
# ── DETERMINE GOALIES TO PROCESS ─────────────────────────────────────────────
 
from pyspark.sql import functions as F
from datetime import datetime, timezone
import json
 
bronze_df = spark.table(BRONZE_TABLE)
 
if not FULL_REFRESH and spark.catalog.tableExists(SILVER_TABLE):
    silver_latest = spark.sql(f"""
        SELECT athlete_id, MAX(silver_ingested_at) AS last_updated
        FROM {SILVER_TABLE}
        GROUP BY athlete_id
    """)
    to_process = (
        bronze_df
        .join(silver_latest, on="athlete_id", how="left")
        .filter(
            F.col("last_updated").isNull() |
            (F.col("pulled_at") > F.col("last_updated"))
        )
        .drop("last_updated")
    )
    print(f"Incremental — goalies to process: {to_process.count()}")
else:
    to_process = bronze_df
    print(f"Full refresh — goalies to process: {to_process.count()}")

In [0]:
# ── LOAD DIM REFERENCES ───────────────────────────────────────────────────────
 
dim_athletes = (
    spark.table(DIM_ATHLETES)
    .filter(
        (F.col("league") == LEAGUE) &
        (F.col("is_current") == True)
    )
    .select(
        F.col("athlete_id").cast("string").alias("dim_athlete_id"),
        F.col("clutch_athlete_id"),
        F.col("clutch_team_id"),
    )
)
 
# Alias conflicting columns to avoid ambiguity
dim_games = (
    spark.table(DIM_GAMES)
    .filter(F.col("league") == LEAGUE)
    .select(
        F.col("clutch_game_id"),
        F.col("source_event_id").alias("dim_event_id"),
        F.col("round").alias("dim_round"),
        F.col("is_playoff").alias("dim_is_playoff"),
        F.col("game_number_in_series"),
        F.col("series_key"),
    )
)
 
print(f"dimAthletes (active, {LEAGUE}) : {dim_athletes.count()}")
print(f"dimGames ({LEAGUE})            : {dim_games.count()}")

In [0]:
# ── UNPACK GAMELOG ROWS ───────────────────────────────────────────────────────
 
silver_ingested_at = datetime.now(timezone.utc).isoformat()
rows               = []
goalies_processed  = 0
goalies_skipped    = 0
 
def parse_toi(toi_str):
    """Convert MM:SS string to integer seconds."""
    try:
        if not toi_str or ":" not in str(toi_str):
            return 0
        parts = str(toi_str).split(":")
        return int(parts[0]) * 60 + int(parts[1])
    except:
        return 0
 
def to_int(val):
    try:
        return int(float(val)) if val not in (None, "", "null") else None
    except:
        return None
 
def to_float(val):
    try:
        return float(val) if val not in (None, "", "null") else None
    except:
        return None
 
def parse_save_pct(val):
    """
    savePct comes as ".880" — add leading zero before casting.
    Returns None for null/empty values.
    """
    try:
        if val in (None, "", "null"):
            return None
        s = str(val).strip()
        if s.startswith("."):
            s = "0" + s
        return float(s)
    except:
        return None
 
bronze_rows = to_process.collect()
 
for br in bronze_rows:
    athlete_id   = br["athlete_id"]
    athlete_name = br["athlete_name"]
    team_abbr    = br["team_abbreviation"]
    team_id      = br["team_id"]
    pulled_at    = br["pulled_at"]
    source_file  = br["source_file"]
    round_num    = br["round"]
 
    # ── Parse names array ──
    names_list = br["names"].split("|") if br["names"] else []
    if not names_list:
        goalies_skipped += 1
        print(f"  SKIPPED (no names): {athlete_name} ({athlete_id})")
        continue
 
    # ── Unpack seasonTypes JSON ──
    try:
        season_types = json.loads(br["season_types_json"])
    except Exception as e:
        goalies_skipped += 1
        print(f"  SKIPPED (JSON parse error): {athlete_name} ({athlete_id}): {e}")
        continue
 
    for st in season_types:
        for cat in st.get("categories", []):
            for event in cat.get("events", []):
                event_id   = event.get("eventId")
                stats      = event.get("stats", [])
                event_type = event.get("type", {})
                type_abbr  = event_type.get("abbreviation", "")
 
                # ── Determine season_type ──
                if type_abbr.startswith("RD") or "round" in event_type.get("slug", ""):
                    season_type_label = "playoffs"
                else:
                    season_type_label = "regular"
 
                # ── Zip names with stats ──
                stat_dict = dict(zip(names_list, stats))
 
                # ── Parse TOI ──
                toi_seconds = parse_toi(stat_dict.get("timeOnIcePerGame"))
 
                # ── Two participation flags ──
                played_in_game = toi_seconds > 0
                started_game   = to_int(stat_dict.get("gameStarted")) == 1
 
                rows.append({
                    # ── Natural keys ──
                    "athlete_id":         athlete_id,
                    "event_id":           event_id,
 
                    # ── Context ──
                    "athlete_name":       athlete_name,
                    "team_abbreviation":  team_abbr,
                    "team_id":            team_id,
                    "season":             SEASON,
                    "season_type":        season_type_label,
                    "round":              to_int(round_num),
                    "is_playoff":         season_type_label == "playoffs",
 
                    # ── Participation flags ──
                    "played_in_game":     played_in_game,
                    "started_game":       started_game,
 
                    # ── Goalie stats ──
                    "toi_seconds":        toi_seconds,
                    "wins":               to_int(stat_dict.get("wins")),
                    "losses":             to_int(stat_dict.get("losses")),
                    "ties":               to_int(stat_dict.get("ties")),
                    "overtime_losses":    to_int(stat_dict.get("overtimeLosses")),
                    "goals_against":      to_int(stat_dict.get("goalsAgainst")),
                    "avg_goals_against":  to_float(stat_dict.get("avgGoalsAgainst")),
                    "shots_against":      to_int(stat_dict.get("shotsAgainst")),
                    "saves":              to_int(stat_dict.get("saves")),
                    "save_pct":           parse_save_pct(stat_dict.get("savePct")),
                    "shutouts":           to_int(stat_dict.get("shutouts")),
 
                    # ── Metadata ──
                    "silver_ingested_at": silver_ingested_at,
                    "source_file":        source_file,
                    "pulled_at":          pulled_at,
                })
 
    goalies_processed += 1
 
print(f"\nGoalies processed  : {goalies_processed}")
print(f"Goalies skipped    : {goalies_skipped}")
print(f"Game rows built    : {len(rows)}")

In [0]:
# ── SPOT CHECK — verify one goalie's rows ─────────────────────────────────────
 
if rows:
    sample_goalie = rows[0]["athlete_name"]
    sample_rows   = [r for r in rows if r["athlete_name"] == sample_goalie]
    playoff_rows  = [r for r in sample_rows if r["is_playoff"]]
 
    print(f"Spot check — {sample_goalie} ({rows[0]['athlete_id']})")
    print(f"Total game rows  : {len(sample_rows)}")
    print(f"Playoff rows     : {len(playoff_rows)}")
    print(f"Regular rows     : {len(sample_rows) - len(playoff_rows)}")
 
    if playoff_rows:
        print(f"\nFirst playoff game row:")
        for k, v in playoff_rows[0].items():
            print(f"  {k:<25} = {v}")
 
    # Verify save_pct parsing
    bad_sv_pct = [r for r in sample_rows if r["save_pct"] is not None
                  and (r["save_pct"] > 1.0 or r["save_pct"] < 0.0)]
    print(f"\nsave_pct out of range (0-1): {len(bad_sv_pct)} {'✓' if not bad_sv_pct else '<-- fix needed'}")

In [0]:
# ── BUILD DATAFRAME + JOIN DIM REFERENCES ────────────────────────────────────────────────────
 
if not rows:
    raise ValueError("No rows built — check parsing above before writing.")
 
silver_df = spark.createDataFrame(rows)
 
# ── Join dimAthletes → clutch_athlete_id ──
silver_df = (
    silver_df
    .join(
        dim_athletes,
        silver_df.athlete_id == dim_athletes.dim_athlete_id,
        how="left"
    )
    .drop("dim_athlete_id")
)
 
# ── Join dimGames → clutch_game_id ──
silver_df = (
    silver_df
    .join(
        dim_games,
        silver_df.event_id == dim_games.dim_event_id,
        how="left"
    )
    .drop("dim_event_id")
)
 
# ── Warn on unmatched joins ──
unmatched_athletes = silver_df.filter(F.col("clutch_athlete_id").isNull()).count()
# Use coalesce for is_playoff since we have both parsed and dim versions
unmatched_games    = silver_df.filter(
    F.col("clutch_game_id").isNull() & F.coalesce(F.col("dim_is_playoff"), F.col("is_playoff"))
).count()
 
print(f"Unmatched athletes         : {unmatched_athletes}  {'✓' if unmatched_athletes == 0 else '<-- investigate'}")
print(f"Unmatched playoff games    : {unmatched_games}     {'✓' if unmatched_games == 0 else '<-- investigate'}")
 
# ── Final column order ──
silver_df = silver_df.select(
    # ── Surrogate FKs ──
    "clutch_athlete_id",
    "clutch_game_id",
 
    # ── Natural keys ──
    "athlete_id",
    "event_id",
 
    # ── Context ──
    "athlete_name",
    "team_abbreviation",
    "team_id",
    "season",
    "season_type",
    F.coalesce(F.col("dim_round"), F.col("round")).alias("round"),
    F.coalesce(F.col("dim_is_playoff"), F.col("is_playoff")).alias("is_playoff"),
    "game_number_in_series",
    "series_key",
 
    # ── Participation flags ──
    "played_in_game",
    "started_game",
 
    # ── Goalie stats ──
    "toi_seconds",
    "wins",
    "losses",
    "ties",
    "overtime_losses",
    "goals_against",
    "avg_goals_against",
    "shots_against",
    "saves",
    "save_pct",
    "shutouts",
 
    # ── Metadata ──
    "silver_ingested_at",
    "source_file",
    "pulled_at",
)
 
print(f"Total rows to write: {silver_df.count()}")

In [0]:
# ── WRITE TO SILVER ───────────────────────────────────────────────────────────
 
table_exists = spark.catalog.tableExists(SILVER_TABLE)
 
if not table_exists:
    (
        silver_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(SILVER_TABLE)
    )
    print(f"Table created: {SILVER_TABLE}")
 
else:
    silver_df.createOrReplaceTempView("new_goalie_logs")
 
    spark.sql(f"""
        MERGE INTO {SILVER_TABLE} AS target
        USING new_goalie_logs AS source
        ON  target.athlete_id = source.athlete_id
        AND target.event_id   = source.event_id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"Merged into existing table: {SILVER_TABLE}")

In [0]:
# ── VALIDATE ─────────────────────────────────────────────────────────────────
 
print("── Playoff rows sample ──")
spark.sql(f"""
    SELECT
        athlete_name,
        team_abbreviation,
        event_id,
        clutch_game_id,
        season_type,
        round,
        game_number_in_series,
        played_in_game,
        started_game,
        toi_seconds,
        wins,
        losses,
        goals_against,
        shots_against,
        saves,
        save_pct,
        shutouts
    FROM {SILVER_TABLE}
    WHERE is_playoff = true
    ORDER BY athlete_name, event_id
    LIMIT 20
""").show(20, truncate=False)

In [0]:
# ── SANITY CHECKS ─────────────────────────────────────────────────────────────
 
checks = spark.sql(f"""
    SELECT
        COUNT(*)                                                        AS total_rows,
        COUNT(DISTINCT athlete_id)                                      AS unique_goalies,
        COUNT(CASE WHEN is_playoff = true  THEN 1 END)                 AS playoff_rows,
        COUNT(CASE WHEN is_playoff = false THEN 1 END)                 AS regular_rows,
        COUNT(CASE WHEN played_in_game = true  THEN 1 END)             AS played_rows,
        COUNT(CASE WHEN started_game = true    THEN 1 END)             AS started_rows,
        COUNT(CASE WHEN shutouts = 1           THEN 1 END)             AS shutout_games,
        COUNT(CASE WHEN clutch_athlete_id IS NULL THEN 1 END)          AS unmatched_athletes,
        COUNT(CASE WHEN clutch_game_id IS NULL
                    AND is_playoff = true THEN 1 END)                  AS unmatched_playoff_games,
        COUNT(CASE WHEN save_pct > 1.0 THEN 1 END)                    AS bad_save_pct,
        COUNT(CASE WHEN save_pct IS NULL
                    AND played_in_game = true THEN 1 END)              AS null_sv_pct_played,
        ROUND(AVG(CASE WHEN is_playoff AND played_in_game
                       THEN save_pct END), 3)                          AS avg_playoff_sv_pct,
        ROUND(AVG(CASE WHEN is_playoff AND played_in_game
                       THEN goals_against END), 2)                     AS avg_playoff_ga
    FROM {SILVER_TABLE}
""")
 
print("Sanity checks:")
checks.show(truncate=False)